# Curve Fitting Assignment Notebook

Goal: recover the hidden parameters `theta`, `M`, and `X` from the CSV points by undoing the rotation/shift in the curve and fitting the remaining wave term.

The notebook follows one main method, then checks the same answer again with a tighter search window to make sure the result is stable.

In [12]:
import csv
import math
from pathlib import Path

DATA_PATH = Path("xy_data.csv")

rows = list(csv.DictReader(DATA_PATH.open(newline="", encoding="utf-8")))
x_obs = [float(row["x"]) for row in rows]
y_obs = [float(row["y"]) for row in rows]
n = len(rows)

print(f"Loaded {n} points")
print(f"x range: {min(x_obs):.6f} -> {max(x_obs):.6f}")
print(f"y range: {min(y_obs):.6f} -> {max(y_obs):.6f}")

Loaded 1500 points
x range: 59.657204 -> 109.231520
y range: 46.032295 -> 69.685510


## Step 2: undo the shift and rotation

The equation becomes much easier after we reverse the geometry:

x - X = cos(theta) * u - sin(theta) * v
y - 42 = sin(theta) * u + cos(theta) * v

After the reverse transform, one coordinate behaves like the forward motion of the curve, and the other coordinate behaves like the wave term `exp(M * u) * sin(0.3 * u)`.

The notebook searches over `theta` and `X`, transforms the points backward, and then estimates `M` from the transformed data.

In [13]:
def transform_points(theta_deg, X):
    theta = math.radians(theta_deg)
    cos_theta = math.cos(theta)
    sin_theta = math.sin(theta)
    u_values = []
    v_values = []
    for x_target, y_target in zip(x_obs, y_obs):
        x_shifted = x_target - X
        y_shifted = y_target - 42.0
        u = x_shifted * cos_theta + y_shifted * sin_theta
        v = -x_shifted * sin_theta + y_shifted * cos_theta
        u_values.append(u)
        v_values.append(v)
    return u_values, v_values

def estimate_M(u_values, v_values):
    filtered_u = []
    filtered_z = []
    sign_mismatch = 0
    for u, v in zip(u_values, v_values):
        sine_term = math.sin(0.3 * u)
        if abs(sine_term) < 1e-8 or abs(v) < 1e-12:
            continue
        ratio = v / sine_term
        if ratio == 0:
            continue
        if ratio < 0:
            sign_mismatch += 1
        filtered_u.append(u)
        filtered_z.append(math.log(abs(ratio)))
    if len(filtered_u) < 20:
        return None
    numerator = sum(u * z for u, z in zip(filtered_u, filtered_z))
    denominator = sum(u * u for u in filtered_u)
    if denominator == 0:
        return None
    M = numerator / denominator
    rmse_v = 0.0
    for u, v in zip(u_values, v_values):
        v_hat = math.exp(M * u) * math.sin(0.3 * u)
        rmse_v += (v - v_hat) ** 2
    rmse_v = math.sqrt(rmse_v / len(u_values))
    penalty = 0.02 * sign_mismatch / max(1, len(filtered_u))
    return M, rmse_v + penalty, rmse_v, sign_mismatch, len(filtered_u)

def score_candidate(theta_deg, X):
    u_values, v_values = transform_points(theta_deg, X)
    estimate = estimate_M(u_values, v_values)
    if estimate is None:
        return None
    M, objective, rmse_v, sign_mismatch, valid_count = estimate
    return objective, theta_deg, X, M, rmse_v, sign_mismatch, valid_count

## Step 3: search for the best `theta` and `X`

This part tries a grid of possible values, transforms the data for each guess, estimates `M`, and keeps the combination with the smallest error.

In [ ]:
def coarse_to_fine_search():
    best = None

    search_stages = [
        {"theta_min": 0.1, "theta_max": 59.9, "theta_steps": 61, "x_min": 0.0, "x_max": 100.0, "x_steps": 61},
        {"theta_span": 2.0, "theta_steps": 31, "x_span": 8.0, "x_steps": 31},
        {"theta_span": 0.4, "theta_steps": 21, "x_span": 2.0, "x_steps": 21},
    ]

    theta_center = 30.0
    x_center = 50.0

    for stage_index, stage in enumerate(search_stages):
        if stage_index == 0:
            theta_min = stage["theta_min"]
            theta_max = stage["theta_max"]
            x_min = stage["x_min"]
            x_max = stage["x_max"]
        else:
            theta_min = max(0.1, theta_center - stage["theta_span"])
            theta_max = min(59.9, theta_center + stage["theta_span"])
            x_min = max(0.0, x_center - stage["x_span"])
            x_max = min(100.0, x_center + stage["x_span"])

        theta_steps = stage["theta_steps"]
        x_steps = stage["x_steps"]

        for theta_i in range(theta_steps):
            theta_deg = theta_min + (theta_max - theta_min) * theta_i / (theta_steps - 1)
            for x_i in range(x_steps):
                X = x_min + (x_max - x_min) * x_i / (x_steps - 1)
                candidate = score_candidate(theta_deg, X)
                if candidate is None:
                    continue
                if best is None or candidate[0] < best[0]:
                    best = candidate

        theta_center = best[1]
        x_center = best[2]
        print(f"Stage {stage_index + 1} best -> objective={best[0]:.8f}, theta={best[1]:.6f}, X={best[2]:.8f}, M={best[3]:.8f}")

    return best

best = coarse_to_fine_search()
print("\nFinal predicted parameters")
print(f"theta = {best[1]:.8f} degrees")
print(f"X     = {best[2]:.8f}")
print(f"M     = {best[3]:.10f}")
print(f"objective = {best[0]:.10f}")
print(f"rmse_v    = {best[4]:.10f}")
print(f"sign_mismatch = {best[5]}")
print(f"valid points   = {best[6]}")

## Step 4: check the transformed residuals

This cell shows how close the transformed points are to the wave model ie residual = actual value - predicted value. Closer the residuals is to 0 means the fit is working well.

In [ ]:
def residual_report(theta_deg, X, M, sample_count=8):
    u_values, v_values = transform_points(theta_deg, X)
    rows_out = []
    abs_errors = []
    for i, (u_value, v_value) in enumerate(zip(u_values, v_values)):
        v_hat = math.exp(M * u_value) * math.sin(0.3 * u_value)
        residual = v_value - v_hat
        abs_errors.append(abs(residual))
        if i < sample_count:
            rows_out.append((i + 1, u_value, v_value, v_hat, residual))
    print("Sample transformed residuals (first few points):")
    for row in rows_out:
        idx, u_value, v_value, v_hat, residual = row
        print(f"{idx:4d}  u={u_value:10.6f}  v={v_value:10.6f}  v_hat={v_hat:10.6f}  residual={residual:+.6f}")
    print(f"Mean abs residual: {sum(abs_errors) / len(abs_errors):.10f}")

residual_report(best[1], best[2], best[3])

Sample transformed residuals (first few points):
   1  u= 36.786655  v= -3.012556  v_hat= -3.012535  residual=-0.000020
   2  u= 22.903768  v=  1.102619  v_hat=  1.102631  residual=-0.000012
   3  u=  6.707971  v=  1.105599  v_hat=  1.105604  residual=-0.000005
   4  u= 31.357837  v=  0.044627  v_hat=  0.044643  residual=-0.000015
   5  u= 52.793353  v= -0.632010  v_hat= -0.631971  residual=-0.000039
   6  u= 15.573466  v= -1.594234  v_hat= -1.594228  residual=-0.000006
   7  u= 27.548569  v=  2.095260  v_hat=  2.095274  residual=-0.000014
   8  u= 41.949695  v=  0.065243  v_hat=  0.065252  residual=-0.000008
Mean abs residual: 0.0000152057


## Step 5: check the answer again in a smaller search box

It searches a tiny area around the best candidate to confirm the answer is stable and not just a lucky guess from the coarse sweep.

In [ ]:
def local_validation(theta_center, x_center):
    best_local = None
    theta_min = max(0.1, theta_center - 0.15)
    theta_max = min(59.9, theta_center + 0.15)
    x_min = max(0.0, x_center - 1.0)
    x_max = min(100.0, x_center + 1.0)

    for theta_i in range(31):
        theta_deg = theta_min + (theta_max - theta_min) * theta_i / 30
        for x_i in range(31):
            X = x_min + (x_max - x_min) * x_i / 30
            candidate = score_candidate(theta_deg, X)
            if candidate is None:
                continue
            if best_local is None or candidate[0] < best_local[0]:
                best_local = candidate
    return best_local

validation = local_validation(best[1], best[2])
print("Validation best")
print(f"theta = {validation[1]:.8f} degrees")
print(f"X     = {validation[2]:.8f}")
print(f"M     = {validation[3]:.10f}")
print(f"objective = {validation[0]:.10f}")
print(f"rmse_v    = {validation[4]:.10f}")
print(f"sign_mismatch = {validation[5]}")
print(f"valid points   = {validation[6]}")

print("\nDifference from coarse-to-fine best")
print(f"d theta = {validation[1] - best[1]:+.10f}")
print(f"d X     = {validation[2] - best[2]:+.10f}")
print(f"d M     = {validation[3] - best[3]:+.10f}")
print(f"d objective = {validation[0] - best[0]:+.10f}")

Validation best
theta = 30.00000000 degrees
X     = 55.00000000
M     = 0.0299999897
objective = 0.0000173717
rmse_v    = 0.0000173717
sign_mismatch = 0
valid points   = 1500

Difference from coarse-to-fine best
d theta = +0.0000000000
d X     = +0.0000000000
d M     = +0.0000000000
d objective = +0.0000000000
